### Load environment variables and import necessary modules

In [2]:
# Load the environment variables
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [76]:
# Import necessary modules
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.vectorstores import OpenSearchVectorSearch
from opensearchpy import OpenSearch
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import time
from langchain_core.prompts import ChatPromptTemplate
from docx import Document

### 1. Load the documents
### 2. Perform document chunking
### 3. Initialize embedding and LLM

In [ ]:
# Load the document
FILE_PATH = r"<LOCATION OF YOUR DOCUMENT>"
doc_loader = PyMuPDFLoader(FILE_PATH)
docs = doc_loader.load()
len(docs)

548

In [8]:
# Perform chunking
doc_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

splitted_docs = doc_splitter.split_documents(docs)

In [4]:
# Initialize embedding
embedding_hf = HuggingFaceEmbeddings(model = "sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2265.97it/s]


In [12]:
# Initialize model
model = ChatGroq(model="openai/gpt-oss-120b")


### Create all three types of indexes - Flat, HNSW, IVF

In [ ]:
# Create FLAT index with OpenSearchCV
client = OpenSearch(
    hosts = [{"host":"localhost", "port": 9200}],
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl = True,
    verify_certs = False
)

FLAT_INDEX_NAME = "langchain-flat-index"

mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "vector_field": {
                "type": "knn_vector",
                "dimension": 768
            },
            "text": {
                "type": "text"
            }
        }
    }
}

if not client.indices.exists(index=FLAT_INDEX_NAME):
    client.indices.create(
        index=FLAT_INDEX_NAME,
        body=mapping
    )

print("Flat index created")


Flat index created


In [ ]:
# Create HNSW index with OpenSearchCV
client = OpenSearch(
    hosts = [{"host":"localhost", "port": 9200}],
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl = True,
    verify_certs = False
)

HNSW_INDEX_NAME = "langchain-hnsw-index"

mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "vector_field": {
                "type": "knn_vector",
                "dimension": 768,
                "method": {
                    "name": "hnsw",
                    "engine": "faiss",
                    "space_type": "l2",
                    "parameters": {
                        "ef_construction": 100,
                        "m": 16
                    }
                }
            },
            "text": {
                "type": "text"
            }
        }
    }
}

if not client.indices.exists(index=HNSW_INDEX_NAME):
    client.indices.create(
        index=HNSW_INDEX_NAME,
        body=mapping
    )

print("HNSW index created")


c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\opensearchpy\connection\http_urllib3.py:214: UserWarning: Connecting to https://localhost:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


HNSW index created


In [ ]:
# Create IVF index with OpenSearchCV
client = OpenSearch(
    hosts = [{"host":"localhost", "port": 9200}],
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl = True,
    verify_certs = False
)

IVF_INDEX_NAME = "langchain-hnsw-index"

mapping = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "vector_field": {
                "type": "knn_vector",
                "dimension": 768,
                "method": {
                    "name": "ivf",
                    "engine": "faiss",
                    "space_type": "l2",
                    "parameters": {
                        "nlist": 128,
                        "nprobes": 8
                    }
                }
            },
            "text": {
                "type": "text"
            }
        }
    }
}

if not client.indices.exists(index=IVF_INDEX_NAME):
    client.indices.create(
        index=IVF_INDEX_NAME,
        body=mapping
    )

print("IVF index created")


c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\opensearchpy\connection\http_urllib3.py:214: UserWarning: Connecting to https://localhost:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IVF index created


### Define 3 different vector stores with difference indexing

In [ ]:
# Define three different vector stores

vector_store_flat = OpenSearchVectorSearch(
    opensearch_url = "http://localhost:9200",
    index_name = FLAT_INDEX_NAME,
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl=True,
    verify_certs=False,
    embedding_function= embedding_hf
)

vector_store_hnsw = OpenSearchVectorSearch(
    opensearch_url = "http://localhost:9200",
    index_name = HNSW_INDEX_NAME,
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl=True,
    verify_certs=False,
    embedding_function= embedding_hf
)

vector_store_ivf = OpenSearchVectorSearch(
    opensearch_url = "http://localhost:9200",
    index_name = IVF_INDEX_NAME,
    http_auth=("admin", "<PUT YOUR PASSWORD>"),
    use_ssl=True,
    verify_certs=False,
    embedding_function= embedding_hf
)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\opensearchpy\connection\http_urllib3.py:214: UserWarning: Connecting to https://localhost:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(


In [37]:
# Add documents with flat index
vector_store_flat.add_documents(splitted_docs, bulk_size=2000)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['9bfd7b1e-4a65-4e7c-a6b7-7b84ea1c5354',
 'f28493fb-44cc-4234-828c-1185285893da',
 '8a9a2b5a-951f-49b4-9bed-efd6fd327926',
 '67c12c74-d3d3-4de6-9834-c9ba47adf01b',
 'd76ee035-3a6e-4d94-9ae4-000d071c9a35',
 'aaaa6118-582d-41bc-b451-0c323b8e0746',
 '49df45ce-34ea-40e4-aa50-9de73172873c',
 'f82ccac9-a2cd-4333-ad9a-1f83cb61fc10',
 'd01f7dd1-a6fa-41a7-a11e-01d3b245bf42',
 'fa9e98b5-bb7b-4fcc-a6a4-c39cfa5ca8bd',
 '29568923-7ddf-4d6c-a3b6-4e9174dd8927',
 '0422b7ba-7c2a-41c7-bf39-5fcb4f98c604',
 '7978c435-b603-4ded-a0cc-57946ce78b99',
 '216af8fd-1b51-467c-aab8-f60e2862cc96',
 '50c221a9-3154-4b37-9934-43e0bbfd3de9',
 '4ad4a8aa-31b8-4011-bd1a-ef55594b8b0c',
 '50dabf58-d515-4151-b283-b4875924e0e6',
 'f4f6a073-ecb7-4eee-abaf-aaa608f4a839',
 'd426c26f-e440-4f03-a71c-85802dc22c25',
 '3aa6bb1b-43f3-4f3e-b672-18e4fb83b8d2',
 'e2b26da9-5464-4e84-8ab2-c61061584896',
 '2a7499b9-4427-404b-94e2-f128fd08bb10',
 '262611eb-f23c-4cd7-8704-f93d4ee3fef1',
 '35c353ee-a9b6-40ed-a5d3-ea89a230217d',
 '3fe5634d-f77a-

In [38]:
# Add documents with HNSW index
vector_store_hnsw.add_documents(splitted_docs, bulk_size=2000)

['47ba0398-583c-4560-9169-aa078f1e860d',
 '41826fba-fe79-4a51-bc87-282f3b757361',
 '12e8775f-42dc-4569-9c1d-3670e1e85c73',
 '005ce20f-0b68-4abf-8fe5-515734d18417',
 '8e4729d7-ace2-4d6d-bbdd-6cd3397ee081',
 'd24e2081-6400-400a-ba65-c3f312cb137e',
 'aafa8081-c9a5-47e6-8259-dc5382656f4e',
 '29da5c71-83ac-4d27-bff8-e873b5028e96',
 '7f1e5c5d-90f1-4977-b83a-3f9b9dfa1053',
 '930a1216-1784-46d2-9d17-1ec52b9951be',
 'be7301fa-a336-4ca6-869a-5e5bbe006a3c',
 '4c8cdf5b-bbf0-46e6-a3de-9e2f88daf519',
 '896becfa-deeb-4a96-a139-894cc0a027c9',
 '3173461e-63f6-471e-b76e-99ab4d4ae019',
 '9ef35c35-c3f2-4bfa-9d23-6903460da3d8',
 'a1f7baf4-87de-44ac-8c94-73a6498e03ac',
 '650d5f61-33d5-406f-b7b1-d430f39894e2',
 '318f3f28-f9b1-402f-bca4-691778b38d3f',
 'f658face-9430-4820-b2be-b5ab7b539375',
 'd302fda5-c0c9-4d65-8c9a-b53ee4c30283',
 '1508655c-1f70-4f68-8b83-0c5e26457aca',
 '2ec2885a-8648-48c8-a7f8-6f96be478c41',
 '4d193dc1-d26a-48d8-85a0-b649038281f6',
 '17cce4dd-ff57-4540-9b31-deeca05d3fdf',
 '646ba9f5-ebf9-

In [39]:
# Add documents with IVF index
vector_store_ivf.add_documents(splitted_docs, bulk_size=2000)

['349a597b-7fed-42a5-8618-c9db451dfe97',
 '586667fe-1942-4c2f-bae7-2785130aed13',
 '1c41649e-b9c3-4269-9c04-ba4e3dcb23b9',
 'f760c0df-d74e-4255-a13e-5cd07609e38e',
 'eedfd06a-8d52-4938-9189-98bec57f8dcb',
 '70711437-2b14-4fad-87f4-5893aabf937a',
 '96d6a2ee-d9cd-4b4e-ae02-d651533ba9a1',
 'f0a5906a-c62e-44a9-9ca8-257860e7cb2d',
 '7d32f2ae-05ed-4f26-ac78-02c3bd5b2d9a',
 '0ce4f010-53eb-43e1-bd5f-beeb3aa781b7',
 'a7c46dc2-8fdc-4fbd-854e-26298cf03b77',
 'bbf40025-ad75-448e-b584-eebc7dd94e8b',
 '3c1337b4-9083-4d24-b18b-d6f8d8bfcfb2',
 '739c2e1b-1550-48bc-862e-056268de5e3e',
 'f3b1aedf-c0b5-43f7-9459-2c6fb6540c70',
 '1565f939-ef9f-4e10-b185-514a5c6a6a41',
 'a5218e31-fdc3-4b0d-aa92-dd00a89c8bb3',
 'fd7bd5c0-528a-44d8-8a53-71c384736524',
 '0b4b286a-3557-4ed2-95ab-5202ab255fa8',
 '1a0a1fba-ff20-4744-bdb9-855b56db2a93',
 'a42c649a-d96f-47f4-b20d-b506a87f5fdf',
 '4f725965-2914-4826-812f-66a4d0eb8432',
 '36739786-f78f-4cf1-995e-5d2e608cb38e',
 'd6c9a7ec-fcc3-4579-91ba-ba5d723c73b7',
 'fcd51eff-0a4c-

### Create Retriever pipeline and test it

In [50]:
# Create the retrievers

retriever_flat = vector_store_flat.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"k":5, "score_threshold": 0.5}
)

retriever_hnsw = vector_store_hnsw.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"k":5, "score_threshold": 0.5}
)

retriever_ivf = vector_store_ivf.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"k":5, "score_threshold": 0.5}
)

In [51]:
# Helper function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [52]:
# Create the pipelines
retriever_flat_pipeline = (
    {
        "context": retriever_flat | format_docs,
        "question": RunnablePassthrough()
    }
)

retriever_hnsw_pipeline = (
    {
        "context": retriever_hnsw | format_docs,
        "question": RunnablePassthrough()
    }
)

retriever_ivf_pipeline = (
    {
        "context": retriever_ivf | format_docs,
        "question": RunnablePassthrough()
    }
)

In [ ]:
# Prompt
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using only provided context.

    Context: {context}
    Question: {question}

    """
)

In [54]:
rag_chain_flat = (
    {
        "context": retriever_flat | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_hnsw = (
    {
        "context": retriever_hnsw | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_ivf = (
    {
        "context": retriever_ivf | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [56]:
# Check which retriever is fastest
query = "What is reinforcement learning?"
start = time.time()
response = retriever_flat.invoke(query)
print("Flat: ", time.time() - start) 


query = "What is reinforcement learning?"
start = time.time()
response = retriever_hnsw.invoke(query)
print("Flat: ", time.time() - start)


query = "What is reinforcement learning?"
start = time.time()
response = retriever_ivf.invoke(query)
print("Flat: ", time.time() - start)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Flat:  2.235499143600464
Flat:  0.6343858242034912
Flat:  0.15412187576293945


### Print Retrieval score

In [58]:
# For flat index
response = vector_store_flat.similarity_search_with_score(
    "What is Reinforcement learning?",
    k = 3
) 

for i, (doc, score) in enumerate(response, start=1):
    print(f"Rank: {i}")
    print(f"Score: {score}")

Rank: 1
Score: 0.6156113
Rank: 2
Score: 0.6132787
Rank: 3
Score: 0.60540646


In [59]:
# For HNSW index
response = vector_store_hnsw.similarity_search_with_score(
    "What is Reinforcement learning?",
    k = 3
) 

for i, (doc, score) in enumerate(response, start=1):
    print(f"Rank: {i}")
    print(f"Score: {score}")

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Rank: 1
Score: 0.6156113
Rank: 2
Score: 0.6156113
Rank: 3
Score: 0.6132787


In [60]:
# For IVF index
response = vector_store_ivf.similarity_search_with_score(
    "What is Reinforcement learning?",
    k = 3
) 

for i, (doc, score) in enumerate(response, start=1):
    print(f"Rank: {i}")
    print(f"Score: {score}")

Rank: 1
Score: 0.6156113
Rank: 2
Score: 0.6156113
Rank: 3
Score: 0.6132787


c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### Reranking using MMR - Maximal Marginal Relevance

In [61]:
# Create the retrievers

retriever_flat = vector_store_flat.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":5, "fetch_k": 20, "lambda_mult": 0.7}
)

retriever_hnsw = vector_store_hnsw.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":5, "fetch_k": 20, "lambda_mult": 0.7}
)

retriever_ivf = vector_store_ivf.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":5, "fetch_k": 20, "lambda_mult": 0.7}
)

In [62]:
# Create the pipelines
retriever_flat_pipeline = (
    {
        "context": retriever_flat | format_docs,
        "question": RunnablePassthrough()
    }
)

retriever_hnsw_pipeline = (
    {
        "context": retriever_hnsw | format_docs,
        "question": RunnablePassthrough()
    }
)

retriever_ivf_pipeline = (
    {
        "context": retriever_ivf | format_docs,
        "question": RunnablePassthrough()
    }
)

In [63]:
# Full RAG pipeline
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using only provided context.

    Context: {context}
    Question: {question}

    """
)

In [64]:
# Prompt
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using only provided context.

    Context: {context}
    Question: {question}

    """
)

In [65]:
# Full RAG pipeline
rag_chain_flat = (
    {
        "context": retriever_flat | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_hnsw = (
    {
        "context": retriever_hnsw | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_ivf = (
    {
        "context": retriever_ivf | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

### Generate output

In [68]:
query = "What is Reinforcement learning?"

In [73]:
# Through flat index
response_flat = rag_chain_flat.invoke(query)
print(response_flat)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Reinforcement learning is a type of machine‑learning paradigm in which an autonomous agent learns how to behave by interacting with its environment and receiving a **reward signal** that indicates how desirable each outcome is.  
Instead of being given a set of labeled examples (as in supervised learning) or merely uncovering hidden structure in data (as in unsupervised learning), the agent must **discover through trial‑and‑error** which actions lead to higher cumulative reward.  

A reinforcement‑learning system consists of several key elements:

* **Policy** – a (often stochastic) mapping from perceived states of the environment to actions; the policy determines the agent’s behavior.  
* **Reward signal** – the scalar feedback that defines the goal of the problem; the agent seeks to maximize the sum of these rewards over time.  
* **Value function** – an estimate of the expected future reward for states (or state‑action pairs), used to guide learning.  
* **Model of the environment**

In [74]:
# Through HNSW index
response_hnsw = rag_chain_hnsw.invoke(query)
print(response_hnsw)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Reinforcement learning is a type of learning in which an **agent interacts with an environment** and learns **from its own experience** rather than from labeled examples.  
The problem is defined by a **reward signal** that specifies the goal, and the agent’s behavior is determined by a **policy** (a mapping from perceived states to actions).  The agent also maintains a **value function** (and optionally a model of the environment) to estimate future reward.  By trial‑and‑error the agent adjusts its policy so as to **maximize the cumulative reward**, using signals such as temporal‑difference errors to guide learning.  Thus, reinforcement learning differs from supervised learning (which relies on external examples) and from unsupervised learning (which seeks hidden structure), focusing instead on learning to act so as to obtain the greatest reward.


In [75]:
# Through IVF index
response_ivf = rag_chain_ivf.invoke(query)
print(response_ivf)

c:\Users\arijit_neha\anaconda3\envs\agentic_revision\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Reinforcement learning is a type of machine‑learning paradigm in which an autonomous agent learns **by interacting with its environment** rather than from a set of labeled examples.  
The agent receives a **reward signal** that defines the goal of the task, and it must discover—through trial‑and‑error—the **policy** (a mapping from perceived states to actions) that maximizes the cumulative reward.  To do this, the agent typically also maintains a **value function** (estimating expected future reward) and may use a **model of the environment**.  Thus, reinforcement learning is the process of learning from one’s own experience in order to maximize a reward signal, distinguishing it from supervised learning (learning from correct examples) and unsupervised learning (finding hidden structure in unlabeled data).


In [ ]:
document = Document()

query = "What is Reinforcement learning?"

document.add_heading(
    'RAG Comparison Report',
    level=1
)

document.add_heading(
    'Query',
    level=2
)

document.add_paragraph(query)

# FLAT
document.add_heading(
    'FLAT Index Response',
    level=2
)

document.add_paragraph(response_flat)

# HNSW
document.add_heading(
    'HNSW Index Response',
    level=2
)

document.add_paragraph(response_hnsw)

# IVF
document.add_heading(
    'IVF Index Response',
    level=2
)

document.add_paragraph(response_ivf)

document.save("rag_comparison_report.docx")

print("DOCX report generated successfully")

DOCX report generated successfully
